In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [3]:
transformer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [4]:
trainset = CIFAR10(root='./data', train=True, download=True, transform=transformer)
testset = CIFAR10(root='./data', train=False, download=True, transform=transformer)

100%|██████████| 170M/170M [00:04<00:00, 39.4MB/s]


In [5]:
trainloader = DataLoader(trainset , batch_size =64 , shuffle=True)
testloader = DataLoader(testset , batch_size =64)


In [6]:
from torch.nn.modules.conv import Conv2d
class CNN(nn.Module):
  def __init__(self):
    super(CNN , self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3 ,32, kernel_size=3 , padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),  #kernel size =2 , strinde 2

        nn.Conv2d(32,64, kernel_size=3 , padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),
        nn.Linear(256,10)

    )
  def forward(self , x):
    x = self.conv_layers(x)
    x = x.view(x.size(0), -1)
    x = self.fc_layers(x)
    return x

In [11]:
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [19]:
# Training the CNN
epochs = 10

model.to(device) # Ensure model is on the correct device

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)  # Move data to GPU
        optimizer.zero_grad()

        output = model.forward(images)  # FP
        loss = criterion(output, labels)  # loss fnx
        loss.backward()  # BP
        optimizer.step()  # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3591171629593501
epoch=2/10 & loss=0.9466533118958973
epoch=3/10 & loss=0.7547011688313521
epoch=4/10 & loss=0.6282034215643583
epoch=5/10 & loss=0.5180557703651736
epoch=6/10 & loss=0.422036485979929
epoch=7/10 & loss=0.33030772682689036
epoch=8/10 & loss=0.25939811076349617
epoch=9/10 & loss=0.1958412535993568
epoch=10/10 & loss=0.16220832830461698


In [20]:
#Evalutuion
correct_labels =0
total_labels =0

model.to(device) # Ensure model is on the correct device

with torch.no_grad():
  for images , labels in testloader:
    images, labels = images.to(device), labels.to(device)  # Move data to GPU
    output = model.forward(images)
    _, predicted = torch.max(output.data , 1)
    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 74.65
